# P05 - Basic RAG

## RAG 사전단계 (저장)
vectorstore 2개를 생성하여 각각 2개의 PDF를 load-split-embed-store
1. load -> PyPDF Loader 사용->(`uv add pypdf langchain-community`)
1. split -> 적절한 split 기준을 찾아서 적용 (우선은 `RecursiveCharacterTextSplitter` 사용)
1. embed -> OpenAI `text-embedding-3-small` 사용
1. store -> `InMemoryVectorStore` 사용하되, 총 2개의 vectorstore 생성해야함. 변수명은
    - `nvda_vectorstore`
    - `googl_vectorstore`

## Agent 구현단계
1개의 Agent에 2개의 Tool 주기
1. `nvda_vectorstore` 를 감싸고 있는 Tool
1. `googl_vectorstore` 를 감싸고 있는 Tool
1. (Optional) `TavilySearchTool` 제공 가능
1. System Prompt 와 Tool Description 을 잘 작성하여 필요한 경우 필요한 Tool 호출하도록 제작

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# RAG파일 로드
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

def pdf_loader(filename:str):
  loader = PyPDFLoader(f"C:/Users/nodecrew/Desktop/{filename}")
  pages = loader.load()

  # docs = [] ## 이미 pypdfloader기능에 다 되어있음
  # for page in pages:
  #   doc = Document(page_content=page.page_content, metadata=page.metadata)
  #   docs.append(doc)
  return pages

google_pdf = pdf_loader('GOOG-10-K-2025.pdf')
nvidia_pdf = pdf_loader('NVDA-10-K-2025.pdf')


C:\Users\nodecrew\AppData\Local\Temp\ipykernel_9524\3341981884.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# RAG 데이터 Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_pdf(docs:Document):
  splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)#쪼개진 chunk의 시작 index

  splits = splitter.split_documents(docs)
  return splits

google_splits = split_pdf(google_pdf)
nvidia_splits = split_pdf(nvidia_pdf)

print(f'{len(google_splits)}개의 조각으로 분리')
print(f'{len(nvidia_splits)}개의 조각으로 분리')

461개의 조각으로 분리
481개의 조각으로 분리


In [16]:
# RAG 데이터 Embed
from langchain_openai import OpenAIEmbeddings

# embedding 담당자
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [17]:
# RAG 데이터 Store
from langchain_core.vectorstores import InMemoryVectorStore

# 벡터스토어 세팅(임베딩)
google_vectorstore = InMemoryVectorStore(embedding=embeddings)
nvidia_vectorstore = InMemoryVectorStore(embedding=embeddings)

# 벡터스토어 저장 (문서조각)
google_vectorstore.add_documents(documents=google_splits)
nvidia_vectorstore.add_documents(documents=nvidia_splits)

['43386ae6-03e7-4358-8a2f-f810e4f9baf5',
 '41040591-c8ae-4b95-9387-71bf0c16171d',
 '315315de-87fc-4de5-ba2c-1922f32bce09',
 '115ffb76-5826-49c1-a9ae-d29477df6536',
 '630865fd-2ea3-4955-b41c-c44e54795d4c',
 '9bc67f1e-2fcc-483d-a9ec-879d49c60afb',
 '85455f72-e5c4-4f03-bf57-577c88d2a2bc',
 '841213a4-3c4e-4a6b-89de-e9ac8d685f5c',
 'c7e22e20-3c58-4f5e-a675-2478ec7733b5',
 '4f2ee0fa-3518-4987-87b8-c7a7a06060f2',
 '8fce4f33-19fd-4d60-8427-1f5acc70e87e',
 '57235657-69e4-4ff1-a335-994d8b1d503b',
 '37d8c188-c45c-45dc-876d-aa432458c00c',
 '44bcf9f8-0f2c-48a9-b96d-ccd74f3f15d5',
 '27ed9870-75c2-4a13-acde-310f7903ba7a',
 'b69a1c78-6500-4e5d-9df8-b3ee1d04688f',
 '947a2454-f512-4be8-9abb-289b083aac3f',
 'b021dfe5-671f-4275-8404-efd7230b1daf',
 'f227805b-489c-43ee-ac08-6acaaceb3549',
 '842b7043-9ca2-4dad-abee-939a046efa91',
 '14c12c48-e16b-4c9c-a372-83e5d08fb4e7',
 'bb4ebd97-0d9d-42fd-94a5-27982a58e952',
 'ac521da5-00e5-4c2b-8e37-371246a4863a',
 '5260416f-173c-45b8-98ce-a5fa10854a6f',
 '0cc59b5d-6b05-

In [ ]:
import os
from langchain.tools import tool
from tavily import TavilyClient

@tool(parse_docstring=True)
def rag_search_google_document(query:str):
    """Alphabet Inc.(Google) FY2026 10-K 연례 보고서 문서에서 관련 정보를 검색합니다.

    Google의 재무 실적(매출, 영업이익), 사업 부문(Google Services, Google Cloud, Other Bets),
    주요 제품 및 서비스(Search, YouTube, Gemini, Android, Waymo 등), 리스크 요인(반독점 규제, AI 경쟁 등),
    공급망 및 설비 투자(CapEx) 현황에 대한 질문이 들어올 때 이 툴을 사용합니다.

    Args:
        query: Alphabet/Google 10-K 보고서에서 찾고자 하는 내용에 대한 자연어 검색어.
    """
    retrieved_docs = google_vectorstore.similarity_search(query, k=4)

    result = '\n---\n'.join(map(lambda doc: doc.page_content, retrieved_docs))
    return result

@tool(parse_docstring=True)
def rag_search_nvidia_document(query:str):
    """NVIDIA FY2026 10-K 연례 보고서 문서에서 관련 정보를 검색합니다.

    NVIDIA의 재무 실적(매출, 이익), 사업 부문(Compute & Networking, Graphics),
    주요 제품(Blackwell, Rubin, GeForce 등), 리스크 요인(미국 대중 수출 규제, H20 재고 관련 손실 등),
    공급망 및 운영 현황에 대한 질문이 들어올 때 이 툴을 사용합니다.

    Args:
        query: NVIDIA 10-K 보고서에서 찾고자 하는 내용에 대한 자연어 검색어.
    """
    retrieved_docs = nvidia_vectorstore.similarity_search(query, k=4)

    result = '\n---\n'.join(map(lambda doc: doc.page_content, retrieved_docs))
    return result

@tool
def search_tavily_tool(prompt:str):
    '''tavily에서 받은 질문에 대해서 조회해보는 tool'''
    tavily_client = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))
    res = tavily_client.search(prompt)
    return res
    
    

In [32]:
from langchain.agents import create_agent

prompt = """너는 기업 분석 및 금융/기술 정보 전문 AI 에이전트이다.
사용자의 질문에 답하기 위해 제공된 툴을 적극적으로 활용하여 정확하고 신뢰할 수 있는 정보를 제공하라.

[툴 사용 규칙]
1. 기업 보고서 우선 조회 (RAG Tools):
   - Alphabet(Google) 관련 재무, 실적, 사업, 제품(Gemini 등) 질문 -> `rag_search_google_document` 사용
   - NVIDIA 관련 재무, 실적, 사업, 제품(Blackwell 등) 질문 -> `rag_search_nvidia_document` 사용
   - 두 기업 비교 질문 -> 두 RAG 툴을 모두 호출하여 데이터 비교

2. 외부 검색 활용 (Web Search Tool):
   - 10-K 보고서에 없는 최신 뉴스, 실시간 시장 반응, 주가, 기타 최신 정보 -> `search_tavily_tool` 사용
   - RAG 툴 검색 결과가 불충분하거나 정보가 없을 경우 보완 목적으로 사용

3. 근거 기반 답변:
   - 반드시 툴을 통해 조회된 사실(Fact)에 기반하여 답변하라.
   - 추측이나 근거 없는 정보 제공은 엄격히 금지한다.
   - 정보 출처(예: Alphabet FY2026 10-K, 최신 웹 검색 등)를 답변에 명시하라.
"""

agent = create_agent(
  model='openai:gpt-4.1-mini',
  tools=[rag_search_google_document,rag_search_nvidia_document,search_tavily_tool],
  system_prompt= prompt,
)

answer = agent.invoke({'messages': {'role':'user', 'content':'구글의 재무 상황에 대해서 알려줘'}})

In [38]:
print(answer['messages'])

[HumanMessage(content='구글의 재무 상황에 대해서 알려줘', additional_kwargs={}, response_metadata={}, id='875baa68-64cf-4b65-909a-1935ed8b2ffe'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 639, 'total_tokens': 673, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b0bb179f09', 'id': 'chatcmpl-ELjzhgc3YoBAFFuqCmo6efhSg9ggz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07fca-decc-7e61-aa90-1811b3037e32-0', tool_calls=[{'name': 'rag_search_google_document', 'args': {'query': 'Alphabet Google FY2026 10-K 재